# Iris Classifier — Mage ML Pipeline (Part C)

**Framework:** Mage AI (block-based ML DAG)  
**Algorithm:** `DecisionTreeClassifier` (scikit-learn)  
**Dataset:** Iris (150 samples, 4 features, 3 classes)

### Pipeline Architecture

```
data_loaders/load_iris.py
        │  DataFrame (150 x 5)
        ▼
transformers/prepare_data.py
        │  train=120 / test=30 split
        ▼
transformers/train_model.py
       / \
      /   \
data_exporters/   transformers/
save_results.py   evaluate_model.py
```


## Block 1 — `load_iris` (Data Loader)

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('.').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from iris_project.data_loaders.load_iris import load_iris

df = load_iris()
print(f'\nShape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'\nClass distribution:')
print(df['target'].value_counts().to_string())


Loaded 150 Iris records
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  target
0                5.1               3.5                1.4               0.2       0
1                4.9               3.0                1.4               0.2       0
2                4.7               3.2                1.3               0.2       0
3                4.6               3.1                1.5               0.2       0
4                5.0               3.6                1.4               0.2       0

Shape: (150, 5)
Columns: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)', 'target']

Class distribution:
target
0    50
1    50
2    50


## Block 2 — `transform` (Prepare Data — 80/20 Split)

In [2]:
from iris_project.transformers.prepare_data import transform

split_df = transform(df)
train = split_df[split_df['split'] == 'train']
test  = split_df[split_df['split'] == 'test']

print(f'Train rows: {len(train)}')
print(f'Test rows : {len(test)}')
print(f'split values: {list(split_df["split"].unique())}')
assert 'split' in split_df.columns
print('test_output assertion: split column present [PASS]')


Train rows: 120 | Test rows: 30
Train rows: 120
Test rows : 30
split values: ['test', 'train']
test_output assertion: split column present [PASS]


## Block 3 — `train_model` (Decision Tree, max_depth=5)

In [3]:
from iris_project.transformers.train_model import train_model

evaluation = train_model(split_df, max_depth=5)

print(f'Model        : DecisionTreeClassifier(max_depth={evaluation["max_depth"]})')
print(f'Train rows   : {evaluation["train_rows"]}')
print(f'Test rows    : {evaluation["test_rows"]}')
print(f'Test accuracy: {evaluation["accuracy"]:.4f}')
print()
assert evaluation['train_rows'] == 120
assert evaluation['test_rows']  == 30
assert 0 <= evaluation['accuracy'] <= 1
print('test_output assertion: train=120, test=30, accuracy in [0,1] [PASS]')


Model        : DecisionTreeClassifier(max_depth=5)
Train rows   : 120
Test rows    : 30
Test accuracy: 1.0000

test_output assertion: train=120, test=30, accuracy in [0,1] [PASS]


## Block 4 — `save_results` (Data Exporter)

In [4]:
from iris_project.data_exporters.save_results import save_results

saved = save_results(evaluation,
                     repo_path=str(PROJECT_ROOT / 'iris_project'))

print(f'Model saved : {saved["model_path"]}')
print(f'Predictions : {saved["predictions_path"]}')
print(f'Evaluation  : {saved["results_path"]}')
print(f'Accuracy    : {saved["accuracy"]:.4f}')


Model saved : /Users/satyamghosh/Downloads/freshcart-pipelines-main/iris_project/models/iris_decision_tree.joblib
Predictions : /Users/satyamghosh/Downloads/freshcart-pipelines-main/iris_project/results/iris_predictions.csv
Evaluation  : /Users/satyamghosh/Downloads/freshcart-pipelines-main/iris_project/results/iris_evaluation.json
Accuracy    : 1.0000


## Block 5 — `evaluate_model` (Confusion Matrix + Metrics)

In [5]:
from iris_project.transformers.evaluate_model import evaluate_model

eval_result = evaluate_model(evaluation,
                             repo_path=str(PROJECT_ROOT / 'iris_project'))

print(f'\nFinal accuracy : {eval_result["accuracy"]:.4f}')
print(f'Labels         : {eval_result["labels"]}')
print(f'Confusion matrix: {eval_result["confusion_matrix"]}')


Held-out test accuracy: 1.0000
Confusion matrix:
           predicted_0  predicted_1  predicted_2
actual_0            10            0            0
actual_1             0            9            0
actual_2             0            0           11
Classification metrics:
       class  precision  recall  f1_score  support
           0        1.0     1.0       1.0     10.0
           1        1.0     1.0       1.0      9.0
           2        1.0     1.0       1.0     11.0
    macro avg        1.0     1.0       1.0     30.0
 weighted avg        1.0     1.0       1.0     30.0
    accuracy       None    None       1.0     30.0

Final accuracy : 1.0000
Labels         : [0, 1, 2]
Confusion matrix: [[10, 0, 0], [0, 9, 0], [0, 0, 11]]


## Cell 6 — Depth Experiment (max_depth 1–5)

In [6]:
import pandas as pd

depth_results = []
for depth in range(1, 6):
    ev = train_model(split_df, max_depth=depth)
    depth_results.append({
        'max_depth': depth,
        'accuracy': round(ev['accuracy'], 4),
        'test_accuracy_passed': ev['accuracy'] >= 0.5,
    })

df_depths = pd.DataFrame(depth_results)
print(df_depths.to_string(index=False))
best = df_depths.loc[df_depths.accuracy.idxmax()]
print(f'\nBest depth: {int(best.max_depth)}  (accuracy={best.accuracy:.4f})')


 max_depth  accuracy  test_accuracy_passed
          1    0.6333                  True
          2    0.9667                  True
          3    1.0000                  True
          4    1.0000                  True
          5    1.0000                  True

Best depth: 3  (accuracy=1.0000)


## Cell 7 — Final Results (iris_evaluation.json)

In [7]:
import json
from pathlib import Path

results_path = PROJECT_ROOT / 'iris_project' / 'results' / 'iris_evaluation.json'
with open(results_path) as f:
    final = json.load(f)

print('iris_evaluation.json:')
print(json.dumps(final, indent=2))


iris_evaluation.json:
{
  "accuracy": 1.0,
  "classification_report": {
    "0": {
      "f1-score": 1.0,
      "precision": 1.0,
      "recall": 1.0,
      "support": 10.0
    },
    "1": {
      "f1-score": 1.0,
      "precision": 1.0,
      "recall": 1.0,
      "support": 9.0
    },
    "2": {
      "f1-score": 1.0,
      "precision": 1.0,
      "recall": 1.0,
      "support": 11.0
    },
    "accuracy": 1.0,
    "macro avg": {
      "f1-score": 1.0,
      "precision": 1.0,
      "recall": 1.0,
      "support": 30.0
    },
    "weighted avg": {
      "f1-score": 1.0,
      "precision": 1.0,
      "recall": 1.0,
      "support": 30.0
    }
  },
  "max_depth": 5,
  "model_path": "/home/src/iris_project/models/iris_decision_tree.joblib",
  "predictions_path": "/home/src/iris_project/results/iris_predictions.csv",
  "test_rows": 30,
  "train_rows": 120
}


## Cell 8 — Block Unit Tests

In [8]:
from sklearn.metrics import accuracy_score

# test_output from train_model
assert evaluation['train_rows'] == 120
assert evaluation['test_rows']  == 30
assert 0 <= evaluation['accuracy'] <= 1
print(f'PASS: test_output  (train={evaluation["train_rows"]}, test={evaluation["test_rows"]})')

# test_accuracy from train_model
assert 1 <= evaluation['max_depth'] <= 5
assert hasattr(evaluation['model'], 'predict')
measured = accuracy_score(evaluation['actual_labels'], evaluation['predictions'])
assert abs(evaluation['accuracy'] - measured) < 1e-12
print(f'PASS: test_accuracy (accuracy={evaluation["accuracy"]:.4f})')

# test_output from prepare_data
assert 'split' in split_df.columns
print('PASS: prepare_data  test_output (split column present)')

print('\nAll Mage block tests passed.')


PASS: test_output  (train=120, test=30)
PASS: test_accuracy (accuracy=1.0000)
PASS: prepare_data  test_output (split column present)

All Mage block tests passed.
